# Анализ лояльности пользователей Яндекс Афиши

Автор Дмитриева Ирина

Выполнен 15.11.2026

**Цель проекта** 

Исследовать поведение пользователей Яндекс Афиши и выяснить, какие характеристики заказов и покупательской активности связаны с повторными покупками. Результаты помогут определить перспективные группы пользователей для маркетинговых акций.

**Задачи**

Загрузить данные о заказах, проверить их качество и провести предобработку.

Создать профили пользователей с характеристиками их первых заказов и покупательской активности.

Изучить распределение пользователей по типам мероприятий, устройствам, регионам и билетным операторам.

Проверить связь выручки, количества билетов и временных характеристик покупок с возвращаемостью пользователей.

Проанализировать взаимосвязи между показателями и сформулировать выводы и рекомендации для маркетинговых акций.

**Описание данных**


Для анализа используются данные о заказах Яндекс Афиши за 2024 год. Они содержат информацию о пользователях, датах заказов, выручке, количестве билетов, типах мероприятий, устройствах, регионах и билетных операторах. Данные выгружаются из базы с помощью SQL. Дополнительно используется таблица курсов валют для перевода выручки из тенге в рубли.

**План работы**

Загрузка данных и их предобработка

Создание профиля пользователя

Исследовательский анализ данных:

Исследование признаков первого заказа и их связи с возвращением на платформу

Исследование поведения пользователей через показатели выручки и состава заказа

Исследование временных характеристик первого заказа и их влияния на повторные покупки

Корреляционный анализ количества покупок и признаков пользователя


Общий вывод и рекомендации

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Параметры подключения в файле .env

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---


In [1]:
# Используйте ячейки типа Code для вашего кода,
# а ячейки типа Markdown для комментариев и выводов

In [2]:
# При необходимости добавляйте новые ячейки для кода или текста

In [3]:
# Библиотеки
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%pip install phik
import phik
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
%pip install python-dotenv
from dotenv import load_dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Подключение
load_dotenv(dotenv_path='.env')

engine = create_engine(os.getenv('DB_CONNECTION_STRING'))

AttributeError: 'NoneType' object has no attribute '_instantiate_plugins'

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [ ]:
# Подключение
db_config = {
    'user': 'praktikum_student',
    'pwd': 'Sdf4$2;d-d30pp',
    'host': 'rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net',
    'port': 6432,
    'db': 'data-analyst-afisha'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string)

In [ ]:
# Запрос из подготовительной задачи
query = '''
SELECT
    p.user_id,
    p.device_type_canonical,
    p.order_id,
    p.created_dt_msk AS order_dt,
    p.created_ts_msk AS order_ts,
    p.currency_code,
    p.revenue,
    p.tickets_count,
    p.created_dt_msk::date -
    LAG(p.created_dt_msk::date) OVER (
        PARTITION BY p.user_id
        ORDER BY p.created_dt_msk
    ) AS days_since_prev,
    p.event_id,
    p.service_name,
    e.event_type_main,
    r.region_name,
    c.city_name
FROM afisha.purchases AS p
INNER JOIN afisha.events AS e ON p.event_id = e.event_id
INNER JOIN afisha.city AS c ON e.city_id = c.city_id
INNER JOIN afisha.regions AS r ON c.region_id = r.region_id
WHERE p.device_type_canonical IN ('mobile', 'desktop')
  AND e.event_type_main != 'фильм'
ORDER BY p.user_id
'''

In [ ]:
# Посмотрим на данные
df = pd.read_sql_query(query, con=engine)
display(df.head())
df.info()
print(df.isna().sum())

In [ ]:
# Изменим тип одного столбца
df['days_since_prev'] = df['days_since_prev'].astype('Int64')
df.info()

После обработки с помощью запросов получен датафрейм для анализа лояльности пользователей Яндекс Афиши. В нем 290611 строк, 14 столбцов. У одного из столбцов был изменен тип, days_since_prev, количество дней от предыдущей покупки пользователя. Теперь данные хранятся в целочисленном формате, так как дни могут быть только целвыми. Этот же столбец - единственный с пропусками, так как для пользователей с одной покупкой значение пропущено. Ничего с пропусками делать не нужно. Далее необходимо привести выручку к одной валюте, проверить категории, дубликаты и выбросы, очистить данные при необходимости.

---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [ ]:
tenge = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')
tenge.head()

In [ ]:
# Курс тенге
tenge['data'] = pd.to_datetime(tenge['data'])
tenge['rate'] = tenge['curs'] / tenge['nominal']
rate = tenge.set_index('data')['rate']

# Тенге в рубли
df['revenue_rub'] = df['revenue']
mask = df['currency_code'] == 'kzt'
df.loc[mask, 'revenue_rub'] = (
    df.loc[mask, 'revenue'] * df.loc[mask, 'order_dt'].map(rate)
)

# Результат
df[['currency_code', 'revenue', 'revenue_rub']].head()

Перевели тенге в рубли и добавили столбец с выручкой в рублях. Сопоставили данные по датам для корректировки и уменьшения пропусков.

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

In [ ]:
# Проверим пропуски и типы столбцов
print(df.isna().sum())
df.info()
# days_since_prev преобразовали выше

In [ ]:
# Числовые столбцы понижаем
for col in ['order_id', 'event_id', 'tickets_count']:
    df[col] = pd.to_numeric(df[col], downcast='integer')
df.info()

In [ ]:
# Проверим значения столбцов
for col in ['device_type_canonical', 'service_name', 'event_type_main', 'currency_code']:
    print(df[col].value_counts())
    
print(df['region_name'].unique())
print(df['city_name'].unique())

In [ ]:
# Значений, которые говорили бы об отсутвии информации не замечено. Проверим дубликаты
# Проверим дубликаты
print(df.duplicated().sum())
print(df.nunique())

# Проверим возможные дубликаты
cols = ['user_id', 'event_id', 'order_ts',
        'revenue', 'tickets_count', 'currency_code']
print('Возможные дубликаты:', df.duplicated(subset=cols).sum())
df[df.duplicated(subset=cols, keep=False)].head(10)

In [ ]:
# Удаляем возможные дубликаты
df = df.drop_duplicates(subset=cols).copy()

print('Осталось заказов:', len(df))
print('Удалено дубликатов:', 290611 - len(df))

Явных дубликатов нет, потенциально нашлось 44 дубликата, которые были очищены, так как время заказов совпадает до секунды, маловроятно один пользователь смог бы оформить такой заказ.

In [ ]:
# Посмотрим на распредление
print(df['revenue_rub'].describe())
print(df['tickets_count'].describe())

In [ ]:
# Проверим отрицательные значения
negative = df[df['revenue_rub'] < 0]
print('Заказы с отрицательными значениями:', len(negative))
print('Доля:', round(len(negative) / len(df) * 100, 2), '%')
print(negative['revenue_rub'].describe())
negative[['order_id', 'currency_code', 'revenue',
          'revenue_rub', 'tickets_count']].head(10)

In [ ]:
# Строим графики
revenue = df['revenue_rub']
fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 6),
    gridspec_kw={'height_ratios': [4, 1]}
)
# Гистограмма
axes[0].hist(revenue, bins=30)
axes[0].set_title('Выручка: гистограмма + диаграмма размаха')
axes[0].set_ylabel('Количество заказов')
# Диаграмма размаха
axes[1].boxplot(revenue, vert=False)
axes[1].set_xlabel('Выручка, руб.')

plt.tight_layout()
plt.show()


tickets = df['tickets_count']
fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 6),
    gridspec_kw={'height_ratios': [4, 1]}
)
# Гистограмма
axes[0].hist(tickets, bins=30)
axes[0].set_title('Количество билетов: гистограмма + диаграмма размаха')
axes[0].set_ylabel('Количество заказов')
# Диаграмма размаха
axes[1].boxplot(tickets, vert=False)
axes[1].set_xlabel('Количество билетов в заказе')

plt.tight_layout()
plt.show()

In [ ]:
# Значения выручки явно нужно фильтровать, но с билетами пока неоднозначно. Проверим их.
# Опредляем 99 перцентиль
tickets_99 = df['tickets_count'].quantile(0.99)

# Перед фильтрацией проверим значения крупных заказов
big_tickets = df[df['tickets_count'] > tickets_99]

print('99 перцентиль:', tickets_99)
print('Крупные заказы:', len(big_tickets))
print('Доля:', round(len(big_tickets) / len(df) * 100, 2), '%')

# Посмотрим самые большие значения
df['tickets_count'].value_counts().sort_index().tail(10)

In [ ]:
# Опредляем 99 перцентиль
revenue_99 = df['revenue_rub'].quantile(0.99)

before = len(df)

# Убираем выбросы и отрицательные значения
df_clean = df[
    (df['revenue_rub'] >= 0) &
    (df['revenue_rub'] <= revenue_99)
].copy()

# Проверяем результат
print('99 перцентиль:', round (revenue_99,2))
print('Исходные данные:', len(df))
print('Очищенные данные:', len(df_clean))
print('Удалено:', before - len(df_clean))
print('Доля:', round((before - len(df_clean)) / before * 100, 2), '%')

После изучения 99 перцентилей пришли к выводу, что стоит провести фильтрацию только для выручки. 99 перцентиль количества билетов в заказе соствил 6, доля крупных заказов 0,06%. Возможно это просто заказы больших групп. В выручке удалили 1,1% значений, связанных с самымии высокими значениями и отрицательными значениями. 99 перцентиль выручки составил 2628,42.

In [ ]:
# Посмотрим на данные выручки после отчистки
revenue = df_clean['revenue_rub']

fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 6),
    gridspec_kw={'height_ratios': [4, 1]}
)

# Гистограмма
axes[0].hist(revenue, bins=30)
axes[0].set_title('Выручка после очистки')
axes[0].set_ylabel('Количество заказов')

# Диаграмма размаха
axes[1].boxplot(revenue, vert=False)
axes[1].set_xlabel('Выручка, руб.')

plt.tight_layout()
plt.show()

In [ ]:
# После фильтрации часть заказов удалилась, исправим количество дней с прошлой покупки
df_clean = df_clean.sort_values(['user_id', 'order_ts'])
df_clean['days_since_prev'] = (
    df_clean.groupby('user_id')['order_dt']
    .diff()
    .dt.days
    .astype('Int64')
)
print('Пропуски:', df_clean['days_since_prev'].isna().sum())
print('Пользователи:', df_clean['user_id'].nunique())

Выше мы нашли пропуски в days_since_prev, что связанно с особенностью данных, так как это количество дней после первой покупки. Есть пользователи только с одной покупкой. Остальные столбцы заполнены полностью. Типы данных проверены и оптимизированы. Столбцы 'order_id', 'event_id', 'tickets_count' понижены, размер используемой памяти сокращен с 33,5 до 29,4 МБ. Все идентификаторы заказов уникальны, полных дубликатов нет. Удалены 44 неявных дубликата.

Распределение выручки неравномерное. Средняя выручка составляет 555,59 рублей, медианная 351,14, а максимальная - более 81 тысячи рублей. Поэтому выручка отфильтрована по 99 перцентилю. Отрицательные значения (381) тоже были удалены, причина их появляения точно неизвестна, скорее всего это ошибки, так как суммы там небольшие и на возвраты это непохоже. Всего удалено 3206 заказов, это 1,1% данных. Осталось 287361 заказов в выборке. Большинство заказов содержит небольшое количество билетов, но встречаются крупные покупки. Проверили пропуски в days_since_prev и пересчитали их после удаления части заказов, теперь там 21838 пропусков.

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


In [ ]:
# Сортируем 
df_clean = df_clean.sort_values(['user_id', 'order_ts'])
# Делаем профиль
profiles = df_clean.groupby('user_id').agg(
    first_order_dt=('order_dt', 'first'),
    last_order_dt=('order_dt', 'last'),
    first_device=('device_type_canonical', 'first'),
    first_region=('region_name', 'first'),
    first_service=('service_name', 'first'),
    first_event_type=('event_type_main', 'first'),
    orders_count=('order_id', 'count'),
    avg_revenue=('revenue_rub', 'mean'),
    avg_tickets=('tickets_count', 'mean'),
    avg_days_between=('days_since_prev', 'mean')
)
profiles.head()

In [ ]:
# Добавляем бинарные признаки
profiles['is_two'] = (profiles['orders_count'] >= 2).astype(int)
profiles['is_five'] = (profiles['orders_count'] >= 5).astype(int)

profiles.info()

Создали профили пользователей. Содержится следующая информация: дата первого и последнего заказа; устройство, с которого был сделан первый заказ; регион, в котором был сделан первый заказ; билетный партнера, к которому обращались при первом заказе;
тип первого посещённого мероприятия; общее количество заказов; средняя выручка с одного заказа в рублях; среднее количество билетов в заказе; среднее время между заказами. Добавлены два новых бинарных признака, совершил ли пользователь 2 или 5 и более заказов.

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

In [ ]:
# Средняя выручка с заказа через профили
avg_revenue = (
    (profiles['avg_revenue'] * profiles['orders_count']).sum()
    / profiles['orders_count'].sum()
)

print('Средняя выручка:', round(avg_revenue, 2))
# Остальные показатели
print('Число пользователей:', len(profiles))
print('Доля пользователей, совершивших 2 и более заказа:',
      round(profiles['is_two'].mean() * 100, 2), '%')
print('Доля пользователей, совершивших 5 и более заказов::',
      round(profiles['is_five'].mean() * 100, 2), '%')

In [ ]:
print(profiles['orders_count'].describe())
print(profiles['avg_tickets'].describe())
print(profiles['avg_days_between'].describe())

В выборке 21838 пользователей, средняя выручка с заказа составляет 518,73 руб. Повторные покупки совершили 61,7% пользователей, а пять и более заказов оформили почти 29%.
Среднее количество заказов на пользователя составляет 13,16, при этом медиана равна 2. Максимальное значение достигает 10168 заказа, могут быть выбросы. Среднее количество билетов составляет 2,74, максимальное - 11. Между покупками в среднем проходит почти 16 дней, медиана составляет 8 дней. Далее изучим возможные выборсы.

In [ ]:
# Перцентили количества заказов и билетов
print(profiles['orders_count'].quantile([0.95, 0.99]))
print(profiles['avg_tickets'].quantile([0.95, 0.99]))

У 95% пользователей не более 31 заказа, у 99% не более 152. При этом максимальное количество заказов достигает 10168. Скорее всего это выброс, далее проведен фильтирацию. По среднему количеству билетов различия меньше: 95 перцентиль составляет 4 билета, 99 - 5, максимум - 11. 

In [ ]:
# Фильтруем по заказам
profiles_clean = profiles[
    profiles['orders_count'] <= 152
].copy()

print('До фильтрации:', len(profiles))
print('После фильтрации:', len(profiles_clean))
print('Удалено пользователей:', len(profiles) - len(profiles_clean))
print('Доля:',
      round((len(profiles) - len(profiles_clean)) / len(profiles) * 100, 2), '%')

In [ ]:
profiles_clean[
    ['orders_count', 'avg_tickets', 'avg_days_between']
].describe()

In [ ]:
# Статистика по количеству заказов
orders_before = profiles['orders_count'].sum()
orders_after = profiles_clean['orders_count'].sum()
removed_orders = orders_before - orders_after

print('Всего заказов:', orders_before)
print('Заказов после фильтрации:', orders_after)
print('Заказов удаленных пользователей:', removed_orders)
print('Доля:', round(removed_orders / orders_before * 100, 2), '%')

После проверки удалены пользователи с аномально большими заказами. Максимум был 10168 заказов, а 99 перцентиль - 152. Для дальнейшего анализа исключены пользователи, превышающие эту границу. Удалено 216 пользователей, в выборке осталось 21622 пользователей. Среднее количество заказов снизилось с 13,16 до 6,50, максимальное  теперь - 152.
Среднее количество билетов практически не изменилось и составляет 2,74. Самый крупный заказ по количеству билетов (11) не выглядят аномально и не удалялся. Это могли быть покупки на компанию. Среднее время между покупками примерно 16 дней. В выборке достаточно пользователей для анализа, данные охватывают пять месяцев 2024 года, покупки с мобильных устройств и с компьютеров, содержится много полезной инфомрации о характеристиках заказов, стоимости, количеству купленных билетов, средним количеством дней, информацию о регионе, жанре и билетом операторе и другие. Это позволит сделать полноценный анализ далее.

После фильтрации исключено 216 пользователей (0,99%), на которых приходилось 146913 заказов, более половины от всего количества (51,12%). Такая высокая активность сильно влияет на показатели. Скорее всего надо изучить такую особенность отдельно, это может быть какая-то ошибка, а может и отдельная группа людей, занимающихся массовыми закупками. Для анализа дальше будем испольщовать очищенные данные.

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


In [ ]:
# Признаки первого
features = [
    'first_event_type',
    'first_device',
    'first_region',
    'first_service'
]

# Считаем количество пользователей и их долю
for col in features:
    table = profiles_clean[col].value_counts().reset_index()
    table.columns = [col, 'users_count']

    table['share_pct'] = (
        table['users_count'] / len(profiles_clean) * 100
    ).round(2)

    print('\n', col)
    display(table)

Пользователи распределены по сегментам неравномерно. Чаще всего первой покупкой становятся билеты на концерты (44,23%). Далее идут категория другое (24,98%) и театр (19,63%). Вместе на эти три категории приходится почти 89% пользователей. Большинство пользователей впервые оформляют заказ с мобильного устройства (82,81%). Среди регионов выделяются Каменевский регион (32,77%) и Североярская область (17,43%). Вместе у них около половины первых заказов. Среди билетных операторов лидирует "Билеты без проблем" (23,9%). На пять крупнейших операторов приходится 72,55% первых заказов.

Основные "точки входа" это концерты, мобильные устройства, Каменевский регион и оператор "Билеты без проблем". При этом у многих регионов и операторов доли среди первых пользователей составляют менее процента.

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
# Средняя доля повторных покупателей
avg_2 = profiles_clean["is_two"].mean() * 100
print("Средняя доля повторных покупателей:", round(avg_2, 2), "%")

# Считаем повторные заказы 
for col in features:
    table = profiles_clean.groupby(col)['is_two'].agg(['count', 'mean'])

    table.columns = ['users_count', 'repeat_pct']
    table['repeat_pct'] = (table['repeat_pct'] * 100).round(2)

    table = table.sort_values('users_count', ascending=False)

    print('\n', col)
    display(table)

In [ ]:
# Построим графики
for col in features:
    table = profiles_clean.groupby(col)["is_two"].agg(["count", "mean"])

    # 10 сегментов с наибольшим количеством пользователей
    table = table.sort_values("count", ascending=False).head(10)
    table = table.sort_values("count")

    plt.figure(figsize=(10, 6))

    plt.barh(table.index, table["mean"] * 100)

    # Средняя доля повторных покупателей
    plt.axvline(
        avg_2,
        color="red",
        linestyle="--",
        label="Среднее"
    )

    plt.title("Повторные покупки по признаку " + col)
    plt.xlabel("Доля пользователей с 2 и более заказами, %")
    plt.xlim(0, 100)
    plt.legend()

    plt.tight_layout()
    plt.show()

Средняя доля пользователей с повторными покупками составляет 61,32%. По типам мероприятий чаще возвращаются пользователи, впервые купившие билеты на выставки (64,16%) и в театр (63,44%). При этом пользователей, которые ходят в театр - в 10 раз больше, 4245 против 413 (среди первых заказов). Пользователи, впервые оформившие заказ с компьютера, чаще совершают повторные покупки (63,83%), чем пользователи мобильных устройств (60,8%). Среди регионов выделяются Широковская область (64,43%), Североярская область (63,83%) и Каменевский регион (62,39%). Среди билетных операторов высокие значения у "Край билетов" (65,2%), "Дом культуры" (64,61%). У самого крупного оператора "Билеты без проблем" доля ниже среднего (60.33%).

Для повторных заказов "точками входа" могут стать театры, покупки с компьютера, Широковская и Североярская области, операторы "Край билетов", "Дом культуры". Но большое количество пользователей в сегменте не всегда означает высокий уровень повторных покупок, как например у оператора "Билеты без проблем". И у небольших по пользователям операторов значения не совсем объективные из-за малой выборки, например "Тех билет", 33.33% повторынх заказов на 3 пользователя. "Лимоны", 50% на 2, и "Зе Бест!" 100% на одного. Учитывать "Зе бест!" как точку входа было бы неверно.

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

In [ ]:
# Проверим гипотезу 1. Сравним спорт и концерты
hyp1 = profiles_clean[
    profiles_clean['first_event_type'].isin(['спорт', 'концерты'])
].groupby('first_event_type')['is_two'].agg(['count', 'mean'])

hyp1.columns = ['users_count', 'repeat_pct']
hyp1['repeat_pct'] = (hyp1['repeat_pct'] * 100).round(2)

display(hyp1)

In [ ]:
# Проверим гипотезу 2. Находим 10 регионов с наибольшим количеством пользователей
top_regions = profiles_clean['first_region'].value_counts().head(10).index
top_users = profiles_clean[
    profiles_clean['first_region'].isin(top_regions)
]

# Пользователи из остальных регионов
other_users = profiles_clean[
    ~profiles_clean['first_region'].isin(top_regions)
]

print('Пользователей из топ-10 регионов:', len(top_users))
print('Доля повторных:', round(top_users['is_two'].mean() * 100, 2), '%')
print('Пользователей из остальных регионов:', len(other_users))
print('Доля повторных:', round(other_users['is_two'].mean() * 100, 2), '%')

# Проверим регионы более детально
region_stats = profiles_clean.groupby('first_region')['is_two'].agg(
    ['count', 'sum', 'mean']
)
region_stats.columns = ['users_count', 'repeat_users', 'repeat_pct']
region_stats = region_stats.sort_values('users_count', ascending=False).head(10)
region_stats['repeat_pct'] = (region_stats['repeat_pct'] * 100).round(2)
region_stats['difference_pp'] = (region_stats['repeat_pct'] - avg_2).round(2)

display(region_stats)

Гипотеза 1 не подтвердилась. 
Среди пользователей, впервые купивших билеты на концерты, повторные покупки совершили 61,82%, а среди покупателей спортивных мероприятий 55,79%. Разница 6% в пользу концертов.

Гипотеза 2 частично подтвердилась. 
В 10 регионах с наибольшим количеством пользователей доля повторных покупателей составляет 62,54%, в остальных регионах - 58,17%. Но если смотреть внимательнее, то в топ-10 регионах по пользователям есть три региона с долей повторных заказов ниже среднего: Озернинский край (на 6 процентных пунктов меньше среднего), Малиновоярский округ (-5), Яблоневская область (~-2).

---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


In [ ]:
# Определим avg_revenue_rub
profiles_clean['avg_revenue_rub'] = (profiles_clean['avg_revenue'] / profiles_clean['avg_tickets'])
print(profiles_clean["avg_revenue_rub"])

In [ ]:
# Сравним распределение средней выручки с билета
plt.figure(figsize=(12, 6))

sns.histplot(
    data=profiles_clean,
    x="avg_revenue_rub",
    hue="is_two",
    bins=100,
    stat="density",
    common_norm=False,
    alpha=0.5
)

plt.xlabel("Средняя выручка с билета, руб.")
plt.ylabel("Плотность распредления")
plt.title("Средняя выручка с билета по группам пользователей")
plt.show()

In [ ]:
# Диапазоны средней выручки с билета
revenue_stats = profiles_clean.groupby('is_two')['avg_revenue_rub'].describe()
display(revenue_stats.round(2))

Средняя выручка с билета у групп с одним заказом и с повторными почти одинаковая, около 200 руб. У пользователей с одним заказом половина значений находится в диапазоне от 56 до 310 руб., а у вернувшихся пользователей от 106 до 276 руб. Среди пользователей с одним заказом больше покупок с очень низкой средней выручкой с билета, но встречаются и более дорогие билеты. У вернувшихся пользователей значения распределены более равномерно и сосредоточены в диапазоне примерно от 100 до 300 руб. Медиана у вернувшихся пользователей выше: 189 руб. против 163 руб, хотя средние значения почти не отличаются.

---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


In [ ]:
# Выбираем пользователей с 2+ заказами
revenue_users = profiles_clean[profiles_clean['orders_count'] >= 2].copy()

# Создаем группы
revenue_users['order_group'] = revenue_users['is_five'].map({
    0: '2-4 заказа',
    1: '5 и более заказов'
})

plt.figure(figsize=(12, 6))

sns.histplot(
    data=revenue_users,
    x='avg_revenue',
    hue='order_group',
    bins=50,
    stat='density',
    common_norm=False,
    alpha=0.5
)

plt.xlabel('Средняя выручка с заказа, руб.')
plt.ylabel('Плотность распределения')
plt.title('Средняя выручка с заказа по группам пользователей')
plt.show()

In [ ]:
# Статистика
revenue_stats = revenue_users.groupby('order_group')['avg_revenue'].describe()
display(revenue_stats.round(2))

Распределения средней выручки с заказа у двух групп отличаются.У пользователей с 2-4 заказами половина значений находится в диапазоне от 220 до 799 руб., а у пользователей с 5 и более диапазон от 332 до 701 руб. В группе с более частыми заказами наблюдается концентрация значений около 500 руб. Средняя выручка с заказа почти одинаковая: 552 руб. у пользователей с 2-4 заказами и 537 руб. у пользователей с 5 и более заказами. Медиана во второй группе немного выше: 513 руб. против 472 руб.

У более активных пользователей средняя выручка с заказа не выше, но они у них больше заказов в одном ценовом диапазоне.

---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

In [ ]:
# Распределение среднего количества билетов
plt.figure(figsize=(10, 5))

sns.histplot(data=profiles_clean, x='avg_tickets', bins=30)

plt.title('Распределение по среднему количеству билетов')
plt.xlabel('Среднее количество билетов в заказе')
plt.ylabel('Количество пользователей')
plt.show()

display(profiles_clean['avg_tickets'].describe().round(2))

In [ ]:
# Создаем сегменты по количеству билетов
bins = [1, 2, 3, 5, float('inf')]
labels = ['1-2 билета', '2-3 билета', '3-5 билетов', '5+ билетов']

profiles_clean['ticket_segment'] = pd.cut(
    profiles_clean['avg_tickets'],
    bins=bins,
    labels=labels,
    right=False
)
# Статистика по сегментам
ticket_stats = profiles_clean.groupby(
    'ticket_segment', observed=True
)['is_two'].agg(['count', 'mean'])

ticket_stats.columns = ['users_count', 'repeat_pct']

ticket_stats['repeat_pct'] = (ticket_stats['repeat_pct'] * 100).round(2)
ticket_stats['share_pct'] = (
    ticket_stats['users_count'] / len(profiles_clean) * 100
).round(2)

display(ticket_stats)

In [ ]:
# Построим график повторных покупок
plt.figure(figsize=(10, 5))

plt.barh(ticket_stats.index.astype(str), ticket_stats['repeat_pct'])

plt.axvline(
    profiles_clean['is_two'].mean() * 100,
    color='red',
    linestyle='--',
    label='Среднее'
)

plt.title('Доля повторных покупателей по количеству билетов')
plt.xlabel('Доля повторных покупателей, %')
plt.xlim(0, 100)
plt.gca().invert_yaxis()
plt.legend()
plt.show()

У большинства покупателей в среднем от 2 до 5 билетов за заказ, на эти сегменты приходится 85,79% пользователей.Самая высокая доля повторных покупателей в сегменте от 2 до 3 билетов (73,56%), что выше среднего по выборке (61,32%). В остальных сегментах доля повторных покупок ниже среднего. Меньше всего пользователей в сегменте от 5 билетов, в котором повторные покупки совершили только 18,76% пользователей. Это самая "аномальная" категория.

Пользователи распределены по сегментам неравномерно. Чаще всего возвращаются пользователи, которые в среднем покупают от 2 до 3 билетов. При большем количестве билетов доля повторных покупателей не увеличивается.

---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
# День недели первой покупки
profiles_clean['first_weekday'] = profiles_clean['first_order_dt'].dt.dayofweek
# Названия дней недели
weekdays = [
    'Понедельник', 'Вторник', 'Среда', 'Четверг',
    'Пятница', 'Суббота', 'Воскресенье'
]

# Статистика по дням недели
weekday_stats = profiles_clean.groupby('first_weekday')['is_two'].agg(['count', 'mean'])

weekday_stats.columns = ['users_count', 'repeat_pct']
weekday_stats['repeat_pct'] = (weekday_stats['repeat_pct'] * 100).round(2)
weekday_stats = weekday_stats.reindex(range(7))
weekday_stats.index = weekdays

display(weekday_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Количество пользователей
axes[0].barh(weekday_stats.index, weekday_stats['users_count'])
axes[0].set_title('Количество пользователей по дням недели')
axes[0].set_xlabel('Количество пользователей')
axes[0].invert_yaxis()

# Доля повторных покупателей
axes[1].barh(weekday_stats.index, weekday_stats['repeat_pct'])
axes[1].axvline(
    profiles_clean['is_two'].mean() * 100,
    color='red',
    linestyle='--',
    label='Среднее'
)
axes[1].set_title('Повторные покупки по дням недели')
axes[1].set_xlabel('Доля повторных покупателей, %')
axes[1].set_xlim(0, 100)
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

Больше всего первых покупок совершено в субботу (3323 пользователя), меньше всего в воскресенье (2777). Самая высокая доля повторных покупателей наблюдается в понедельник (62,99%) и субботу (62,77%), самая низкая в четверг (59,45%) и пятницу (59,77%). Разница между максимальным и минимальным показателем составляет 3,54 процентного пункта. В целом значения близки к среднему по выборке (61,32%).

Количество пользователей по дням недели распределено достаточно равномерно. Связи между днем недели первой покупки и повторными заказами не обнаружено.

---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


In [ ]:
# Группы пользователей
group_2_4 = profiles_clean[
    (profiles_clean['orders_count'] >= 2) &
    (profiles_clean['orders_count'] <= 4)
]

group_5 = profiles_clean[profiles_clean['orders_count'] >= 5]
print('2-4 заказа:', round(group_2_4['avg_days_between'].mean(), 2))
print('5 и более заказов:', round(group_5['avg_days_between'].mean(), 2))

print('Медиана 2-4 заказа:', round(group_2_4['avg_days_between'].median(), 2))
print('Медиана 5 и более заказов:', round(group_5['avg_days_between'].median(), 2))

In [ ]:
plt.figure(figsize=(12, 6))

bins = range(0, 151, 5)

plt.hist(group_2_4['avg_days_between'].dropna(),
         bins=bins, density=True, alpha=0.5, label='2-4 заказа')

plt.hist(group_5['avg_days_between'].dropna(),
         bins=bins, density=True, alpha=0.5, label='5 и более заказов')

plt.title('Интервал между заказами')
plt.xlabel('Средний интервал между заказами, дни')
plt.ylabel('Плотность распределения')
plt.legend()
plt.show()

У пользователей с 2-4 заказами средний интервал между покупками составляет 21,44 дня, а у пользователей с 5 и более заказами 9,98 дня. Медианы отличаются незначительно: 9 и 8,22 дня соответственно. У пользователей с 2-4 заказами чаще встречаются большие интервалы между покупками.

Пользователи с 5 и более заказами в среднем совершают покупки чаще. Однако по этим данным нельзя утверждать, что короткий интервал сам по себе повышает вероятность повторной покупки, ведь мы рассматриваем пользователей, которые уже совершили повторный заказ.

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

Сначала рассчитаем коэффициенты корреляции phi_k между количеством заказов и характеристиками пользователей. Используем данные из профилей пользователей. Посмотрим, какие признаки имеют наиболее высокую связь с количеством заказов.

In [ ]:
# Признаки для анализа
cols = [
    'first_device',
    'first_region',
    'first_service',
    'first_event_type',
    'first_weekday',
    'avg_revenue',
    'avg_revenue_rub',
    'avg_tickets',
    'avg_days_between',
    'orders_count'
]

corr_data = profiles_clean[cols].copy()

# Приведем столбец к обычному float
corr_data['avg_days_between'] = corr_data['avg_days_between'].astype(float)

# Числовые признаки
interval_cols = [
    'avg_revenue',
    'avg_revenue_rub',
    'avg_tickets',
    'avg_days_between',
    'orders_count'
]

# Матрица корреляций
phik_matrix = corr_data.phik_matrix(interval_cols=interval_cols)

# Корреляция признаков с количеством заказов
display(
    phik_matrix['orders_count']
    .sort_values(ascending=False)
    .round(2)
)

In [ ]:
# Тепловая карта
plt.figure(figsize=(12, 8))

sns.heatmap(
    phik_matrix,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    vmin=0,
    vmax=1
)

plt.title('Коэффициент корреляции phi_k для анализируемых показателей')
plt.tight_layout()
plt.show()

Наибольшая связь с количеством заказов наблюдается у среднего интервала между покупками (0,29), среднего количества билетов (0,23) и средней выручки с заказа (0,22). Но значения корреляции невысоки.

При этом на тепловой карте есть сильные связи между самими признаками. Например, между регионом и билетным оператором коэффициент составляет 0,70, а между оператором и типом мероприятия 0,59. Средний интервал между заказами нужно рассмотреть отдельно, так как у пользователей с одним заказом этот показатель отсутствует.

In [ ]:
# Распределение количества заказов, у меня это значение хранится в orders_count
display(profiles_clean['orders_count'].value_counts().sort_index().head(10))
display(profiles_clean['orders_count'].describe())

Проверили распределение количества заказов. Больше всего пользователей совершили только один заказ (8363 человека). При увеличении количества заказов число пользователей в основном снижается. Медиана заказов 2, а серднее - 6.

Чтобы проверить, изменятся ли результаты корреляционного анализа, разделим пользователей на три сегмента: с одним заказом, с 2-4 заказами и с 5 и более заказами.

In [ ]:
# Создаем сегменты по количеству заказов
profiles_clean['orders_segment'] = pd.cut(
    profiles_clean['orders_count'],
    bins=[0, 1, 4, float('inf')],
    labels=['1 заказ', '2-4 заказа', '5 и более заказов']
)

display(profiles_clean['orders_segment'].value_counts())

In [ ]:
# Данные для корреляции с сегментами
seg_data = corr_data.drop(
    columns=['orders_count', 'avg_days_between']
).copy()

seg_data['orders_segment'] = profiles_clean['orders_segment']
# Коэффициенты
seg_phik = seg_data.phik_matrix(
    interval_cols=['avg_revenue', 'avg_revenue_rub', 'avg_tickets']
)

# Корреляция phi_k
display(seg_phik['orders_segment'].sort_values(ascending=False).round(2))

In [ ]:
# Тепловая карта
plt.figure(figsize=(12, 8))

sns.heatmap(
    seg_phik,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    vmin=0,
    vmax=1
)

plt.title('Корреляция phi_k с сегментами по количеству заказов')
plt.tight_layout()
plt.show()

Первый анализ показал слабую связь большинства признаков с количеством заказов. Наиболее высокие коэффициенты были у среднего интервала между покупками (0,29), среднего количества билетов (0,23) и средней выручки с заказа (0,22). После разделения пользователей на три сегмента, высокие коэффициенты в выборке получили среднее количество билетов в заказе (0,38), средняя выручка с заказа (0,33) и средняя выручка с билета (0,29). Для устройства, типа мероприятия, билетного оператора и дня недели коэффициенты остаются низкими.

Количество заказов сильнее связано со средним количеством билетов и выручкой.

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

Информация о данных.

Для анализа использовали данные о заказах Яндекс Афиши за 2024 год. Изначально было 290611 заказов. Выручку в тенге перевели в рубли, проверили пропуски, дубликаты и выбросы. Удалили 44 возможных дубликата, отрицательную выручку и значения выше 99 перцентиля. После очистки остался 287361 заказ.

Затем создали профили пользователей и исключили 216 человек с аномально большим количеством заказов. В итоговую выборку вошли 21622 пользователя. Важно учитывать, что на исключенных пользователей приходилось 51,12% всех заказов, поэтому результаты анализа не отражают поведение самых активных покупателей и этот момент требует отдельного исследования. Это могут быть как ошибки, так и пользоватли, которые массово скупают билеты (перекупщики, турфирмы и тд).

Основные результаты анализа.

В итоговой выборке 61,32% пользователей совершили два и более заказа, а 28,28% оформили пять и более заказов. У 8363 пользователей был только один заказ. В среднем на пользователя приходится 6,5 заказа, медиана составляет 2 заказа.

Средняя выручка с заказа до фильтрации самых активных пользователей составляла 518,73 руб. Среднее количество билетов в заказе составляет около 2,74. Чаще всего пользователи впервые покупают билеты на концерты (44,23%) и оформляют заказ с мобильного устройства (82,81%).

Средняя доля повторных покупателей составляет 61,32%. Выше среднего показатели у пользователей, впервые купивших билеты в театр (63,44%) и на выставки (64,16%), а также оформивших первый заказ с компьютера (63,83%). 

Гипотеза о том, что покупатели спортивных мероприятий возвращаются чаще покупателей концертов, не подтвердилась: 55,79% против 61,82%.В десяти крупнейших регионах доля повторных покупателей выше, чем в остальных: 62,54% против 58,17%. Однако в трех регионах из топ-10 показатель ниже среднего по выборке.

Средняя выручка с билета у пользователей с одним заказом и с повторными покупками почти одинаковая, около 200 руб. При этом медиана у вернувшихся пользователей немного выше: 189 руб. против 163 руб. У пользователей с 2-4 заказами средняя выручка с заказа составляет 552 руб., а у пользователей с 5 и более заказами 537 руб. Существенного роста средней выручки с увеличением количества заказов не наблюдается. Самая высокая доля повторных покупателей у пользователей, которые в среднем покупают от 2 до 3 билетов (73,56%). В группе с 5 и более билетами показатель значительно ниже (18,76%).

День недели первой покупки слабо связан с повторными заказами. Самая высокая доля возвратов наблюдается в понедельник (62,99%), самая низкая в четверг (59,45%). Разница составляет 3,54 процентного пункта.

У пользователей с 2-4 заказами средний интервал между покупками составляет 21,44 дня, а у пользователей с 5 и более заказами 9,98 дня. Более активные покупатели обычно возвращаются быстрее, хотя медианы интервалов у двух групп близки.

Анализ phi_k показал, что с количеством заказов сильнее связаны выручка и среднее количество билетов. Наиболее высокие коэффициенты были у среднего интервала между покупками (0,29), среднего количества билетов (0,23) и средней выручки с заказа (0,22). После разделения пользователей на три сегмента, высокие коэффициенты получили среднее количество билетов в заказе (0,38), средняя выручка с заказа (0,33) и средняя выручка с билета (0,29). Для устройства, типа мероприятия, билетного оператора и дня недели коэффициенты остаются низкими.

Рекомендации.

Уделить больше внимания покупателям концертов и театра. Концерты привлекают больше всего пользователей, а у театрального сегмента доля повторных покупок выше среднего. Можно проверить специальные предложения и подборки похожих мероприятий для таких клиентов.

Поработать над удержанием мобильных пользователей. Через мобильные устройства приходит 82,81% пользователей, но доля повторных покупок у них ниже, чем у пользователей компьютеров. Стоит проверить удобство повторного заказа и протестировать рекомендации.

Обратить внимание на покупателей спортивных мероприятий и крупных заказов. У них доля повторных покупок ниже среднего. Можно проверить, помогут ли напоминания о новых мероприятиях.

Протестировать предложения для повторных покупок. У активных пользователей средний интервал между заказами составляет около 10 дней. Можно проверить эффективность напоминаний и подборок мероприятий через 10 дней после покупки.

Отдельно изучить пользователей с аномально большим количеством заказов. Их немного, но на них приходится больше половины заказов до фильтрации профилей. Нужно выяснить причины такой активности и выработать стратегию для них.

### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.

**Ссылка на проект**

https://github.com/iraldmitr/ds_m2_final_project